## Sentiment Analysis

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("junaid6731/hospital-reviews-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'hospital-reviews-dataset' dataset.
Path to dataset files: /kaggle/input/hospital-reviews-dataset


In [2]:
import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import precision_score, recall_score, matthews_corrcoef

In [3]:
import os

DATA_PATH = path

print(os.listdir(DATA_PATH))

['hospital.csv']


In [4]:
df = pd.read_csv(DATA_PATH + "/hospital.csv")
print(df.head())
print(df.shape)

                                            Feedback  Sentiment Label  \
0  Good and clean hospital. There is great team o...                1   
1  Had a really bad experience during discharge. ...                1   
2  I have visited to take my second dose and Proc...                1   
3   That person was slightly clueless and offered...                1   
4  There is great team of doctors and good OT fac...                0   

   Ratings  Unnamed: 3  
0        5         NaN  
1        5         NaN  
2        4         NaN  
3        3         NaN  
4        1         NaN  
(996, 4)


In [5]:
df = df.drop(columns=["Unnamed: 3"], errors="ignore")

In [6]:
df = df.rename(columns={
    "Feedback": "text",
    "Sentiment Label": "label"
})

In [7]:
df = df.dropna(subset=["text", "label"])

In [8]:
print(df.head())
print(df["label"].value_counts())

                                                text  label  Ratings
0  Good and clean hospital. There is great team o...      1        5
1  Had a really bad experience during discharge. ...      1        5
2  I have visited to take my second dose and Proc...      1        4
3   That person was slightly clueless and offered...      1        3
4  There is great team of doctors and good OT fac...      0        1
label
1    728
0    268
Name: count, dtype: int64


In [9]:
model_name = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

In [10]:
dataset = Dataset.from_pandas(df)

In [11]:
def tokenize(batch):

    return tokenizer(
        batch["text"],
        padding=True,
        truncation=True,
        max_length=128
    )


dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/996 [00:00<?, ? examples/s]

In [12]:
dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

In [13]:
dataset = dataset.rename_column("label", "labels")

dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [14]:
training_args = TrainingArguments(

    output_dir="./bert_patient_reviews",

    eval_strategy="epoch",

    per_device_train_batch_size=8,   # small dataset
    per_device_eval_batch_size=8,

    num_train_epochs=5,   # more epochs (small data)

    learning_rate=2e-5,

    logging_steps=20,

    save_strategy="epoch",

    load_best_model_at_end=True,

    report_to="none"
)

In [15]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=dataset["train"],
    eval_dataset=dataset["test"]
)

In [16]:
trainer.train()

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


RuntimeError: `fused=True` requires all the params to be floating point Tensors of supported devices: ['mps', 'cuda', 'xpu', 'hpu', 'cpu', 'mtia', 'privateuseone'] but torch.float32 and xla

In [ ]:
# Already: model.save_pretrained('sentimentmodeldir')
# Zip it:
!zip -r sentiment_model.zip sentimentmodeldir/
print("✅ BERT zipped!")


In [ ]:
preds = trainer.predict(dataset["test"])

y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)

In [ ]:
from sklearn.metrics import precision_score, recall_score, matthews_corrcoef, accuracy_score

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
mcc = matthews_corrcoef(y_true, y_pred)
accuracy = accuracy_score(y_true, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("MCC      :", mcc)

In [ ]:
def predict_sentiment(text):

    device = next(model.parameters()).device   # detect GPU/CPU

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Move inputs to same device as model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

        probs = torch.softmax(outputs.logits, dim=1)

    return probs.cpu().numpy()   # move back to CPU for printing

In [ ]:
review = "Doctors were careless and nurses were rude"

print(predict_sentiment(review))

In [ ]:
def predict_sentiment_label(text):

    probs = predict_sentiment(text)[0]

    if probs[1] > probs[0]:
        return "Positive Review", probs
    else:
        return "Negative Review", probs


print(predict_sentiment_label(review))

In [ ]:
save_path = "./clinicalbert_model"

tokenizer.save_pretrained(save_path)
model.save_pretrained(save_path)

**Pickling**